# Lab 2 — The toolbox for images: convolutional networks

In Lab 1 we classified images with an MLP: `nn.Flatten` turned each image into a vector of 784 pixels, throwing away
the information about **which pixels are neighbours**. Moreover, every neuron of the first layer was connected to
every pixel: lots of parameters.

Convolutional networks (CNNs) fix both problems. In this notebook we look at their building blocks, one by one:

1. `nn.Conv2d` — the convolutional layer
2. Pooling — max, average and adaptive pooling
3. `nn.BatchNorm2d` — batch normalization
4. `nn.Dropout` — regularization
5. Putting it all together (and tracking the shapes!)

Afterwards, in `cnn_mnist.py`, you will build your own CNN with these pieces.

In [ ]:
import matplotlib.pyplot as plt
import torch
from torch import nn
from torchvision import datasets
from torchvision.transforms import ToTensor

torch.manual_seed(0)

# MNIST: 28x28 grayscale images of handwritten digits (~10 MB download the first time)
mnist = datasets.MNIST(root="data", train=True, download=True, transform=ToTensor())

img, label = mnist[0]
x = img.unsqueeze(0)          # add the batch dimension (→ tensors §6)
print(f"Label: {label}, image shape: {tuple(x.shape)}  (batch, channels, H, W)")

plt.figure(figsize=(3, 3))
plt.imshow(img.squeeze(), cmap="gray")
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()

💡 Convolutional layers in PyTorch always expect images in the format $N \times C \times H \times W$
(batch, channels, height, width), even for a single image (→ tensors §2).

---
## 1. `nn.Conv2d` — the convolutional layer

A convolution slides a small **filter** (kernel), e.g. $3 \times 3$, over the image. At each position it computes a
weighted sum of the pixels under the filter: the output is a **feature map** that says "how much of this pattern is
here". Key ideas:
- **local connectivity**: each output looks only at a small neighbourhood;
- **weight sharing**: the same filter is used at every position, so few parameters.

Main arguments: `in_channels`, `out_channels` (= number of filters), `kernel_size`, `stride`, `padding`.

In [ ]:
conv = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, padding=1)

print("Weight shape:", tuple(conv.weight.shape), " (out_channels, in_channels, kH, kW)")
print("Bias shape:  ", tuple(conv.bias.shape))

out = conv(x)
print("Output shape:", tuple(out.shape))   # 8 feature maps, same 28x28 size thanks to padding=1

### What does a filter detect?

The filters are **learned** during training. To get an intuition, let's set two filters by hand: the classic
**Sobel** filters, which detect vertical and horizontal edges.

In [ ]:
sobel = torch.tensor([[-1., 0., 1.],
                      [-2., 0., 2.],
                      [-1., 0., 1.]])

edge_conv = nn.Conv2d(1, 2, kernel_size=3, padding=1, bias=False)
with torch.no_grad():                  # we are setting weights by hand, not training (→ tensors §13.8)
    edge_conv.weight[0, 0] = sobel     # vertical edges
    edge_conv.weight[1, 0] = sobel.T   # horizontal edges

edges = edge_conv(x).detach()

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(img.squeeze(), cmap="gray")
axes[0].set_title("input")
axes[1].imshow(edges[0, 0], cmap="RdBu")
axes[1].set_title("vertical edges")
axes[2].imshow(edges[0, 1], cmap="RdBu")
axes[2].set_title("horizontal edges")
for ax in axes:
    ax.axis("off")
plt.show()

And these are the 8 feature maps produced by the **random** (untrained) filters of `conv`: after training they would
become detectors of useful patterns.

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axes):
    ax.imshow(out[0, i].detach(), cmap="gray")
    ax.set_title(f"map {i}")
    ax.axis("off")
plt.show()

### Output size

For each spatial dimension: $\quad H_{out} = \left\lfloor \dfrac{H_{in} + 2p - k}{s} \right\rfloor + 1$

- `padding=1` with `kernel_size=3` and `stride=1` keeps the size unchanged ("same" convolution);
- `stride=2` halves it.

In [ ]:
for k, s, p in [(3, 1, 0), (3, 1, 1), (5, 1, 2), (3, 2, 1)]:
    layer = nn.Conv2d(1, 4, kernel_size=k, stride=s, padding=p)
    print(f"kernel={k} stride={s} padding={p}  ->  {tuple(layer(x).shape)}")

### Why so few parameters?

Let's compare a convolution with 32 filters and a linear layer producing an output of the same size:

In [ ]:
conv32 = nn.Conv2d(1, 32, kernel_size=3, padding=1)            # output 32 x 28 x 28
n_conv = sum(p.numel() for p in conv32.parameters())
n_linear = 28 * 28 * (32 * 28 * 28) + 32 * 28 * 28              # Linear(784, 25088): weights + biases
print(f"Conv2d(1, 32, 3):             {n_conv:>12,} parameters")
print(f"Linear with same output size: {n_linear:>12,} parameters")

---
## 2. Pooling

A pooling layer **reduces the spatial size** of the feature maps. It slides a window over each feature map (e.g.
$2 \times 2$ with stride 2) and replaces each region with a **single summary value**. What changes between the
variants is only *how* the region is summarized.

Common properties of all pooling layers:
- they have **no parameters**: nothing to learn;
- they act on **each channel separately**: the number of channels does not change, only $H$ and $W$;
- why use them: fewer values → less computation in the following layers; each subsequent filter "sees" a larger
  portion of the original image; and a bit of robustness to small shifts of the input.

We will use this small $4 \times 4$ "feature map" as a running example:

In [ ]:
t = torch.tensor([[1., 3., 2., 1.],
                  [4., 6., 5., 0.],
                  [7., 2., 9., 8.],
                  [1., 0., 3., 4.]]).view(1, 1, 4, 4)   # (batch, channels, H, W)
print(t[0, 0])

### 2.1 Max pooling — `nn.MaxPool2d`

Keeps the **maximum** of each region: "was this pattern present *somewhere* in this area?". It preserves the strongest
activations, which is why it is the most common choice inside the convolutional blocks.

In [ ]:
max_pool = nn.MaxPool2d(kernel_size=2)   # 2x2 windows, stride = kernel_size by default
print(max_pool(t)[0, 0])                 # the max of each 2x2 block

In [ ]:
pooled = max_pool(edges)
print(f"{tuple(edges.shape)} -> {tuple(pooled.shape)}")

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(edges[0, 0], cmap="RdBu")
axes[0].set_title("28 x 28")
axes[1].imshow(pooled[0, 0], cmap="RdBu")
axes[1].set_title("after MaxPool2d(2): 14 x 14")
for ax in axes:
    ax.axis("off")
plt.show()

### 2.2 Average pooling — `nn.AvgPool2d`

Takes the **mean** of each region: a smoother summary that takes all the values into account, not only the strongest.
Same arguments as `MaxPool2d`.

In [ ]:
avg_pool = nn.AvgPool2d(kernel_size=2)
print("max:\n", max_pool(t)[0, 0])
print("avg:\n", avg_pool(t)[0, 0])       # e.g. top-left block: (1 + 3 + 4 + 6) / 4 = 3.5

# The same comparison on the feature map of vertical edges
fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(edges[0, 0], cmap="RdBu")
axes[0].set_title("input 28 x 28")
axes[1].imshow(max_pool(edges)[0, 0], cmap="RdBu")
axes[1].set_title("MaxPool2d(2)")
axes[2].imshow(avg_pool(edges)[0, 0], cmap="RdBu")
axes[2].set_title("AvgPool2d(2)")
for ax in axes:
    ax.axis("off")
plt.show()

### 2.3 Adaptive pooling — `nn.AdaptiveMaxPool2d` / `nn.AdaptiveAvgPool2d`

With max and average pooling we choose the **window** and the output size depends on the input. Adaptive pooling
works the other way round: we choose the **output size**, and PyTorch computes the windows needed to get there.
This makes the network work with input images of **any size**.

In [ ]:
adaptive = nn.AdaptiveMaxPool2d(output_size=2)

# On our 4x4 example it behaves exactly like MaxPool2d(2)...
print(adaptive(t)[0, 0])

# ...but it produces a 2x2 output whatever the input size
for size in [4, 7, 28, 100]:
    print(f"input {size}x{size}  ->  output {tuple(adaptive(torch.rand(1, 1, size, size)).shape[-2:])}")

The most important special case is **global average pooling**, `nn.AdaptiveAvgPool2d(1)`: each whole feature map is
averaged into a single number. Modern CNNs (e.g. ResNet) use it at the end instead of flattening large feature maps:
a network with $C$ channels in its last block produces exactly $C$ features, whatever the input image size.

In [ ]:
gap = nn.AdaptiveAvgPool2d(1)
print(f"{tuple(out.shape)} -> {tuple(gap(out).shape)}")               # (1, 8, 28, 28) -> (1, 8, 1, 1)
print("Same as the mean over H and W?", torch.allclose(gap(out).flatten(1), out.mean(dim=(2, 3))))

---
## 3. `nn.BatchNorm2d` — batch normalization

During training the distribution of the activations keeps changing, which makes optimization harder. Batch norm
**normalizes each channel** to mean 0 and standard deviation 1, computing the statistics over the batch (and over
all the pixels), then applies a learnable scale $\gamma$ and shift $\beta$.
In practice it makes training faster and more stable. It goes after a convolution: `Conv → BatchNorm → ReLU`.

In [ ]:
batch = torch.stack([mnist[i][0] for i in range(32)])   # a batch of 32 images (→ tensors §7)
feats = conv(batch).detach() * 5 + 3                     # feature maps with an arbitrary scale and offset

bn = nn.BatchNorm2d(num_features=8)                      # one (gamma, beta) pair per channel
normed = bn(feats).detach()

print("Per-channel mean before:", feats.mean(dim=(0, 2, 3))[:4])
print("Per-channel mean after: ", normed.mean(dim=(0, 2, 3))[:4])
print("Per-channel std after:  ", normed.std(dim=(0, 2, 3))[:4])
print("\nLearnable parameters:", [name for name, _ in bn.named_parameters()])

⚠️ **Train vs eval.** At test time we may classify a single image, so we cannot compute batch statistics. During
training, BatchNorm keeps a **running average** of mean and variance, and in `model.eval()` mode it uses those
instead. This is why calling `model.train()` / `model.eval()` matters (→ training §4.4)!

In [ ]:
print("Running mean (updated during training):", bn.running_mean[:4])

bn.eval()
print("Eval mode: same input, different output?", not torch.allclose(bn(feats), normed))
bn.train()

---
## 4. `nn.Dropout` — regularization

Dropout randomly zeroes a fraction `p` of the activations during training (and rescales the others by
$1/(1-p)$), so the network cannot rely too much on any single neuron: it reduces **overfitting**.
In `eval()` mode it does nothing. (`nn.Dropout2d` zeroes whole feature maps instead of single values.)

In [ ]:
drop = nn.Dropout(p=0.5)
a = torch.ones(10)

drop.train()
print("train:", drop(a))
drop.eval()
print("eval: ", drop(a))

---
## 5. Putting it all together

A typical CNN stacks a few **convolutional blocks** `Conv → BatchNorm → ReLU → MaxPool`: at each block the images
get **smaller** (pooling) but **deeper** (more channels). At the end we flatten and use a linear layer to produce
the 10 class scores.

In [ ]:
model = nn.Sequential(
    # block 1: 1 x 28 x 28 -> 16 x 14 x 14
    nn.Conv2d(1, 16, kernel_size=3, padding=1),
    nn.BatchNorm2d(16),
    nn.ReLU(),
    nn.MaxPool2d(2),
    # block 2: 16 x 14 x 14 -> 32 x 7 x 7
    nn.Conv2d(16, 32, kernel_size=3, padding=1),
    nn.BatchNorm2d(32),
    nn.ReLU(),
    nn.MaxPool2d(2),
    # classifier
    nn.Flatten(),
    nn.Dropout(0.25),
    nn.Linear(32 * 7 * 7, 10),
)

🔍 **The most common error** when building a CNN is getting the input size of the first `nn.Linear` wrong.
The trick: pass a fake batch through the layers one at a time and print the shapes.

In [ ]:
h = batch[:4]
print(f"{'input':12s} -> {tuple(h.shape)}")
for layer in model:
    h = layer(h)
    print(f"{layer.__class__.__name__:12s} -> {tuple(h.shape)}")

💡 Alternatively, `nn.LazyLinear(10)` works out its input size automatically the first time it is called.

Compare the number of parameters with the MLP of Lab 1 (669,706 parameters):

In [ ]:
print(f"CNN parameters: {sum(p.numel() for p in model.parameters()):,}")

About 30 times fewer parameters... and, as you will see in the exercise, it is also much more accurate.

---
## Summary

| Layer | What it does | Changes the shape? | Parameters |
|---|---|---|---|
| `nn.Conv2d(C_in, C_out, k, stride, padding)` | learns `C_out` local filters | channels → `C_out`; H, W by the formula above | yes |
| `nn.BatchNorm2d(C)` | normalizes each channel | no | $\gamma, \beta$ (+ running stats) |
| `nn.ReLU()` | nonlinearity | no | no |
| `nn.MaxPool2d(2)` | max over 2×2 regions | H, W halved | no |
| `nn.AvgPool2d(2)` | mean over 2×2 regions | H, W halved | no |
| `nn.AdaptiveMaxPool2d(s)` / `nn.AdaptiveAvgPool2d(s)` | pooling to a fixed output size | H, W → s | no |
| `nn.AdaptiveAvgPool2d(1)` | global average of each map | H, W → 1 | no |
| `nn.Dropout(p)` / `nn.Dropout2d(p)` | randomly zeroes activations (train only) | no | no |
| `nn.Flatten()` | $C \times H \times W$ → vector | yes | no |
| `nn.Linear(in, out)` / `nn.LazyLinear(out)` | final classifier | yes | yes |

👉 **Now it's your turn:** open `cnn_mnist.py` and build your network in the `TODO`.